In [1]:
import pandas as pd

# Start your engines Vroom Vroom
#from sqlalchemy import create_engine
#engine = create_engine("sqlite:///../sql/streaming.db")


#  CHECK database
from sqlalchemy import create_engine, text
from pathlib import Path
import os

BASE_DIR = Path.cwd().parent   # project root
DB_PATH = BASE_DIR / "sql" / "streaming.db"


print("Notebook CWD:", os.getcwd())
print("DB exists?", DB_PATH.exists())

# Correct way: string + SQLite URL scheme
engine = create_engine(f"sqlite:///{DB_PATH}", echo=False)

with engine.connect() as conn:
    tables = conn.execute(text("SELECT name FROM sqlite_master WHERE type='table';")).fetchall()
    print("Tables in DB:", tables)

Notebook CWD: /Users/jessica/PortfolioProjects/Streaming_Project/notebooks
DB exists? True
Tables in DB: [('titles',), ('movies',), ('tv_shows',), ('genres',), ('title_genres',)]


In [2]:
# MESSING AROUND
#Titles added per year to test it all out

#query = """
#SELECT release_year, COUNT(*) as titles_added
#FROM titles
#GROUP BY release_year
#ORDER BY release_year DESC
#LIMIT 20
#"""

#pd.read_sql(query, engine)

In [3]:
#pd.read_sql("PRAGMA table_info(tv_shows);", engine)

In [4]:
# MESSING AROUND

#query="""
#SELECT titles.title, tv_shows.seasons, titles.date_added, titles.release_year   
#FROM titles
#INNER JOIN tv_shows ON titles.title_id = tv_shows.title_id 
#ORDER BY seasons DESC
#LIMIT 20
#"""

#pd.read_sql(query, engine)

In [5]:
# MESSING AROUND

#query="""
#SELECT titles.title, movies.runtime_minutes   
#FROM titles
#INNER JOIN movies ON titles.title_id = movies.title_id 
#ORDER BY runtime_minutes DESC
#LIMIT 20
#"""

#pd.read_sql(query, engine)

In [6]:
# PROJECT 
# number of titles in each genre total

query="""
WITH genre_count AS (
    SELECT g.genre, 
        COUNT (DISTINCT tg.title_id) AS num
    FROM genres g
    JOIN title_genres tg ON g.genre_id = tg.genre_id
    GROUP BY g.genre_id
    ORDER BY num DESC
)

SELECT genre, num
FROM genre_count gc
"""

pd.read_sql(query, engine)

,genre,num
0,International Movies,2752
1,Dramas,2427
2,Comedies,1674
3,International TV Shows,1351
4,Documentaries,869
5,Action & Adventure,859
6,TV Dramas,763
7,Independent Movies,756
8,Children & Family Movies,641
9,Romantic Movies,616


In [7]:
# IMPORTANT: GENRES IN ANIME TITLES


query="""

-- Step 1: Identify all titles classified as Anime (Features or Series)
-- This defines the population we want to analyze.

WITH anime_titles AS( 
    SELECT tg.title_id
    FROM title_genres tg
    JOIN genres g ON g.genre_id = tg.genre_id
    WHERE g.genre IN ('Anime Features','Anime Series')
),

-- Step 2: Reference only those titles back to the title_genres table
-- so we can see ALL genres attached to Anime titles (many-to-many expansion).

anime_genres AS(
    SELECT tg.genre_id, tg.title_id
    FROM title_genres tg
    JOIN anime_titles ant ON ant.title_id = tg.title_id
)

-- Step 3: Count how many Anime titles appear in each non-Anime genre
-- This shows which genres most commonly co-occur with Anime.

SELECT g.genre,
    COUNT(DISTINCT ag.title_id) AS num
FROM anime_genres ag
JOIN genres g ON ag.genre_id = g.genre_id
WHERE g.genre NOT IN ('Anime Features','Anime Series')
GROUP BY g.genre
ORDER BY num DESC
LIMIT 20
"""

pd.read_sql(query, engine)

,genre,num
0,International TV Shows,126
1,Action & Adventure,50
2,International Movies,46
3,Kids' TV,24
4,Crime TV Shows,16
5,Teen TV Shows,15
6,Children & Family Movies,14
7,Romantic TV Shows,14
8,TV Thrillers,8
9,Sci-Fi & Fantasy,7


In [8]:
# IMPORTANT: NUMBER OF TITLES IN EACH GENRES THAT ARE NOT ANIME


query = """
WITH non_anime_titles AS( 
    SELECT DISTINCT tg.title_id
    FROM title_genres tg 
    WHERE NOT EXISTS (
        SELECT 1  -- existance check
        FROM title_genres tg2
        JOIN genres g2 ON g2.genre_id = tg2.genre_id
        WHERE tg2.title_id = tg.title_id
        AND g2.genre IN ('Anime Features','Anime Series')
    )
),

non_anime_genres AS(
    SELECT tg.genre_id, tg.title_id
    FROM title_genres tg
    JOIN non_anime_titles nat ON nat.title_id = tg.title_id
)

SELECT g.genre,
    COUNT(DISTINCT nag.title_id) AS num
FROM non_anime_genres nag
JOIN genres g ON nag.genre_id = g.genre_id
GROUP BY g.genre
ORDER BY num DESC;

"""

pd.read_sql(query, engine)

,genre,num
0,International Movies,2706
1,Dramas,2427
2,Comedies,1674
3,International TV Shows,1225
4,Documentaries,868
5,Action & Adventure,809
6,TV Dramas,763
7,Independent Movies,756
8,Children & Family Movies,627
9,Romantic Movies,613


In [9]:
query="""


--Verifying that this split as it was supposed to. 8560 < 8807 (see next block)

WITH non_anime_titles AS( 
    SELECT DISTINCT tg.title_id
    FROM title_genres tg 
    WHERE NOT EXISTS (
        SELECT 1  -- existance check
        FROM title_genres tg2
        JOIN genres g2 ON g2.genre_id = tg2.genre_id
        WHERE tg2.title_id = tg.title_id
        AND g2.genre IN ('Anime Features','Anime Series')
    )
)

SELECT COUNT(*) 
FROM non_anime_titles;

"""
pd.read_sql(query, engine)



,COUNT(*)
0,8560


In [10]:
query="""

SELECT COUNT(DISTINCT title_id)
FROM title_genres;

"""
pd.read_sql(query, engine)

,COUNT(DISTINCT title_id)
0,8807


In [11]:
query="""
SELECT 
    COUNT (tg.title_id) as num_titles,
    g.genre
FROM title_genres tg
JOIN genres g ON tg.genre_id = g.genre_id
WHERE g.genre IN ('Anime Features','Anime Series')
GROUP BY tg.genre_id
LIMIT 20
"""
pd.read_sql(query, engine)

,num_titles,genre
0,71,Anime Features
1,176,Anime Series


In [12]:
#query="""
#SELECT *   
#FROM titles
#WHERE listed_in LIKE '%Horror%'
#LIMIT 20
#"""


query="""
SELECT titles.title, movies.runtime_minutes   
FROM titles
INNER JOIN movies ON titles.title_id = movies.title_id 
ORDER BY runtime_minutes DESC
LIMIT 20
"""

query="""
SELECT titles.title, titles.listed_in, tv_shows.seasons
FROM titles
INNER JOIN tv_shows ON titles.title_id = tv_shows.title_id 
WHERE listed_in LIKE '%Horror%'
AND titles.type = "TV Show"
ORDER BY tv_shows.seasons DESC
LIMIT 20
"""


pd.read_sql(query, engine)

,title,listed_in,seasons
0,Supernatural,"Classic & Cult TV, TV Action & Adventure, TV H...",15
1,American Horror Story,"TV Horror, TV Mysteries, TV Thrillers",9
2,IZombie,"TV Comedies, TV Dramas, TV Horror",5
3,Z Nation,"TV Action & Adventure, TV Comedies, TV Horror",5
4,The Originals,"TV Dramas, TV Horror, TV Mysteries",5
5,Bates Motel,"Crime TV Shows, TV Dramas, TV Horror",5
6,Haven,"Classic & Cult TV, TV Horror, TV Mysteries",5
7,Lost Girl,"TV Dramas, TV Horror, TV Mysteries",5
8,Castlevania,"Anime Series, TV Horror, TV Thrillers",4
9,Chilling Adventures of Sabrina,"TV Horror, TV Mysteries, TV Sci-Fi & Fantasy",4


In [13]:
query="""
SELECT
    g.genre,
    COUNT(DISTINCT t.title_id) AS titles_count
FROM titles t
JOIN title_genres tg ON t.title_id = tg.title_id
JOIN genres g ON tg.genre_id = g.genre_id
WHERE g.genre IN ('Anime Series', 'Anime Feature')
GROUP BY g.genre;
"""
pd.read_sql(query, engine)

,genre,titles_count
0,Anime Series,176


In [14]:
# with CTE, make list of anime titles
# get genres and title count where title is in anime titles
# result is a list of genre counts for anime titles
# in other words, what genres of anime are available.



query="""


WITH anime_titles AS(

SELECT t.title_id
FROM titles t
JOIN title_genres tg
    ON t.title_id = tg.title_id
JOIN genres g 
    ON tg.genre_id = g.genre_id
WHERE g.genre IN ('Anime Series', 'Anime Feature')
)

SELECT
    g.genre,
    COUNT(a.title_id) AS titles_count
FROM anime_titles a
JOIN title_genres tg
    ON a.title_id = tg.title_id
JOIN genres g
    ON tg.genre_id = g.genre_id
GROUP BY g.genre
ORDER BY titles_count DESC;
"""

pd.read_sql(query, engine)

,genre,titles_count
0,Anime Series,176
1,International TV Shows,126
2,Kids' TV,24
3,Crime TV Shows,16
4,Teen TV Shows,15
5,Romantic TV Shows,14
6,TV Thrillers,8
7,TV Horror,6
8,Spanish-Language TV Shows,1
9,Stand-Up Comedy & Talk Shows,1


In [15]:



query="""
SELECT COUNT(titles.title) AS num_titles, genres.genre
FROM titles
JOIN title_genres ON title_genres.title_id = titles.title_id
JOIN genres ON title_genres.genre_id = genres.genre_id
WHERE genres.genre = 'Anime Series'
LIMIT 20;
"""
pd.read_sql(query, engine)

,num_titles,genre
0,176,Anime Series


In [16]:
query="""
SELECT genres.genre
FROM genres
"""

pd.read_sql(query, engine)

,genre
0,Action & Adventure
1,Anime Features
2,Anime Series
3,British TV Shows
4,Children & Family Movies
5,Classic & Cult TV
6,Classic Movies
7,Comedies
8,Crime TV Shows
9,Cult Movies


In [19]:
query="""

WITH anime_titles AS (
    SELECT DISTINCT tg.title_id
    FROM title_genres tg
    JOIN genres g 
        ON g.genre_id = tg.genre_id
    WHERE g.genre IN ('Anime Series', 'Anime Features')
)

SELECT 
    tg.title_id,
    g.genre,
    CASE 
        WHEN at.title_id IS NOT NULL THEN 1
        ELSE 0
    END AS is_anime
FROM title_genres tg
JOIN genres g 
    ON g.genre_id = tg.genre_id
LEFT JOIN anime_titles at 
    ON tg.title_id = at.title_id
"""

pd.read_sql(query, engine)

,title_id,genre,is_anime
0,1,Documentaries,0
1,2,International TV Shows,0
2,2,TV Dramas,0
3,2,TV Mysteries,0
4,3,Crime TV Shows,0
...,...,...,...
19318,8806,Children & Family Movies,0
19319,8806,Comedies,0
19320,8807,Dramas,0
19321,8807,International Movies,0
